# Modelo Final XGBoost para Tarjetas

## Título y objetivo

Este notebook reconstruye y evalúa el **modelo final XGBoost** seleccionado previamente en `notebooks/4_model_selection_cards.ipynb`. Su propósito es documentar paso a paso la configuración final del modelo que se utilizará como referencia en la tesis, sin abrir una nueva fase de exploración ni comparación de algoritmos.

**Alcance metodológico**

- No se comparan nuevos modelos.
- No se recalcula la selección de variables.
- No se reajustan hiperparámetros con `test`.
- No se optimiza nuevamente el threshold usando `test`.
- Se utiliza exclusivamente la configuración ganadora ya fijada en notebook 4.


El modelo final corresponde a una tarea de clasificación binaria a nivel transaccional, donde:

- una fila representa una transacción
- la variable objetivo es `is_fraud`
- la métrica principal es **PR-AUC / average precision**

El enfoque de este notebook es deliberadamente más estrecho que el de los notebooks 3 y 4: aquí solo se reconstruye el **XGBoost final seleccionado** para dejar trazabilidad empírica, outputs exportables y un texto académico reutilizable en el capítulo de resultados.


## Imports


In [1]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.metrics import classification_report, confusion_matrix

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (path / "src").exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")


## Conexión con src


In [2]:
from src.models.final_xgboost_model import (
    build_configuration_table,
    build_feature_importance_frame,
    build_final_metrics_table,
    build_final_model_artifact,
    build_split_summary,
    compute_train_validation_caps,
    get_dataset_schema,
    load_final_split,
    load_final_xgboost_inputs,
    load_train_validation_frame,
    make_default_final_xgboost_config,
    predict_final_model,
    train_final_xgboost_model,
    validate_required_files,
    validate_selected_features,
)
from src.utils.metrics import get_predicted_labels
from src.utils.notebook_common import ensure_project_root_on_path, save_table_with_optional_excel
from src.utils.plotting import (
    plot_confusion_matrix,
    plot_feature_importance,
    plot_precision_recall_curve,
    plot_probability_distribution,
    plot_roc_curve,
)

PROJECT_ROOT = ensure_project_root_on_path(PROJECT_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PROJECT_ROOT = C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models


## Definición de rutas


In [3]:
config = make_default_final_xgboost_config(PROJECT_ROOT)

MODELING_DATASET_PATH = config.modeling_data_path
SELECTED_FEATURES_PATH = config.selected_features_path
FINAL_MODEL_COMPARISON_PATH = config.final_model_comparison_path
THRESHOLD_OPTIMIZATION_PATH = config.threshold_optimization_path
BEST_MODEL_METADATA_PATH = config.best_model_metadata_path
BEST_MODEL_FINAL_ARTIFACT_PATH = config.best_model_final_artifact_path
SOURCE_MODEL_ARTIFACT_PATH = config.source_model_artifact_path
FINAL_METRICS_SUMMARY_PATH = config.final_metrics_summary_path
TABLES_DIR = config.tables_dir
FIGURES_DIR = config.figures_dir
MODELS_DIR = config.models_dir

required_files = validate_required_files(config)
display(required_files)


,artifact_name,path,exists,size_mb
0,modeling_dataset,C:\Users\cecor\OneDrive - Universidad Nacional...,True,3287.6184
1,selected_features,C:\Users\cecor\OneDrive - Universidad Nacional...,True,0.0228
2,final_model_comparison,C:\Users\cecor\OneDrive - Universidad Nacional...,True,0.0008
3,threshold_optimization,C:\Users\cecor\OneDrive - Universidad Nacional...,True,0.0069
4,best_model_metadata,C:\Users\cecor\OneDrive - Universidad Nacional...,True,0.0033
5,best_model_final_artifact,C:\Users\cecor\OneDrive - Universidad Nacional...,True,1.2751
6,source_model_artifact,C:\Users\cecor\OneDrive - Universidad Nacional...,True,1.2587
7,final_metrics_summary,C:\Users\cecor\OneDrive - Universidad Nacional...,True,0.0004


## Carga de datos


In [4]:
dataset_columns = get_dataset_schema(MODELING_DATASET_PATH)
split_summary = build_split_summary(config)

dataset_overview = pd.DataFrame(
    [
        {
            "dataset_path": str(MODELING_DATASET_PATH),
            "n_rows_total": int(split_summary["n_rows"].sum()),
            "n_columns_total": len(dataset_columns),
            "target_column_present": config.target_col in dataset_columns,
            "year_month_present": "year_month" in dataset_columns,
            "datetime_present": "datetime" in dataset_columns,
        }
    ]
)

target_distribution = pd.DataFrame(
    [
        {
            "scope": "full_dataset_including_excluded_periods",
            "n_rows": int(split_summary["n_rows"].sum()),
            "n_frauds": int(split_summary["n_positive"].sum()),
            "fraud_rate": float(split_summary["n_positive"].sum() / split_summary["n_rows"].sum()),
        },
        {
            "scope": "train_validation_test_only",
            "n_rows": int(split_summary.loc[split_summary["split"].isin(["train", "validation", "test"]), "n_rows"].sum()),
            "n_frauds": int(split_summary.loc[split_summary["split"].isin(["train", "validation", "test"]), "n_positive"].sum()),
            "fraud_rate": float(
                split_summary.loc[split_summary["split"].isin(["train", "validation", "test"]), "n_positive"].sum()
                / split_summary.loc[split_summary["split"].isin(["train", "validation", "test"]), "n_rows"].sum()
            ),
        },
    ]
)

display(dataset_overview)
display(target_distribution)
display(pd.DataFrame({"dataset_columns_preview": dataset_columns[:50]}))


,dataset_path,n_rows_total,n_columns_total,target_column_present,year_month_present,datetime_present
0,C:\Users\cecor\OneDrive - Universidad Nacional...,24386900,217,True,True,True


,scope,n_rows,n_frauds,fraud_rate
0,full_dataset_including_excluded_periods,24386900,29757,0.001220
1,train_validation_test_only,23761875,29757,0.001252


,dataset_columns_preview
0,user
1,card
2,user_card_id
3,datetime
4,year_month
5,merchant_name
6,merchant_city
7,merchant_state
8,zip
9,mcc


## Carga de configuración ganadora


In [5]:
inputs = load_final_xgboost_inputs(config)
metadata = inputs["metadata"]
hyperparameters = inputs["hyperparameters"]

winner_summary = pd.DataFrame(
    [
        {
            "model_name": metadata["model_name"],
            "feature_subset": metadata["feature_subset"],
            "balancing_strategy": metadata["balancing_strategy"],
            "threshold_final": inputs["threshold_final"],
            "threshold_from_source_artifact": inputs["threshold_from_source_artifact"],
            "random_state": inputs["random_state"],
            "train_validation_row_cap": inputs["train_validation_row_cap"],
            "training_data_rule": metadata["training_data"],
            "xgboost_version_current_env": xgb.__version__,
        }
    ]
)

hyperparameters_df = pd.DataFrame(
    [{"hyperparameter": key, "value": value} for key, value in hyperparameters.items()]
)

display(
    Markdown(
        f'''
        **Fuente de verdad metodológica**

        - Artefacto final exacto de notebook 4: `{BEST_MODEL_FINAL_ARTIFACT_PATH.name}`
        - Metadata final: `{BEST_MODEL_METADATA_PATH.name}`
        - Threshold final de validación: `{inputs["threshold_final"]:.2f}`
        - Threshold heredado del artefacto base del experimento: `{inputs["threshold_from_source_artifact"]:.2f}`

        En este notebook se usa como referencia oficial el artefacto final ya materializado en notebook 4, porque es el objeto exacto con el que se reportaron las métricas finales del modelo seleccionado.
        '''
    )
)

display(winner_summary)
display(hyperparameters_df)



        **Fuente de verdad metodológica**

        - Artefacto final exacto de notebook 4: `best_model_final.joblib`
        - Metadata final: `best_model_metadata.json`
        - Threshold final de validación: `0.71`
        - Threshold heredado del artefacto base del experimento: `0.41`

        En este notebook se usa como referencia oficial el artefacto final ya materializado en notebook 4, porque es el objeto exacto con el que se reportaron las métricas finales del modelo seleccionado.
        

,model_name,feature_subset,balancing_strategy,threshold_final,threshold_from_source_artifact,random_state,train_validation_row_cap,training_data_rule,xgboost_version_current_env
0,xgboost,top_100,scale_pos_weight_manual,0.71,0.41,42,500000,train + validation (combined with model-specif...,3.2.0


,hyperparameter,value
0,objective,binary:logistic
1,eval_metric,logloss
2,n_estimators,300
3,max_depth,6
4,learning_rate,0.05
5,subsample,0.8
6,colsample_bytree,0.8
7,gamma,None
8,min_child_weight,None
9,reg_alpha,None


## Validación de variables seleccionadas


In [6]:
feature_validation = validate_selected_features(
    dataset_columns=dataset_columns,
    selected_features=inputs["selected_features"],
    target_col=config.target_col,
)

selected_features_final = inputs["selected_features_table"].copy()
selected_features_final = selected_features_final[selected_features_final["selected_top_100"] == True].copy()
selected_features_final["feature_order_final"] = selected_features_final["feature"].map(
    {feature: idx + 1 for idx, feature in enumerate(inputs["selected_features"])}
)
selected_features_final = selected_features_final.sort_values("feature_order_final").reset_index(drop=True)

feature_validation_df = pd.DataFrame(
    [
        {"validation_check": "n_selected_features", "value": feature_validation["n_selected_features"]},
        {"validation_check": "missing_features", "value": feature_validation["missing_features"]},
        {"validation_check": "duplicated_features", "value": feature_validation["duplicated_features"]},
        {"validation_check": "identifier_hits", "value": feature_validation["identifier_hits"]},
        {"validation_check": "direct_target_hits", "value": feature_validation["direct_target_hits"]},
        {"validation_check": "is_valid", "value": feature_validation["is_valid"]},
    ]
)

if not feature_validation["is_valid"]:
    raise ValueError(f"Las variables seleccionadas no son válidas: {feature_validation}")

display(feature_validation_df)
display(selected_features_final.head(20))


,validation_check,value
0,n_selected_features,100
1,missing_features,[]
2,duplicated_features,[]
3,identifier_hits,[]
4,direct_target_hits,[]
5,is_valid,True


,feature,feature_type,missing_rate,unique_values,mutual_info_score,logistic_abs_coef,tree_importance,permutation_importance_mean,permutation_importance_std,rank_mutual_info,rank_logistic,rank_tree,rank_permutation,rank_average,aggregate_score,constant_filter_status,correlation_filter_status,max_abs_correlation,included_in_modeling,selected_top_25,selected_top_50,selected_top_75,selected_top_100,feature_order_final
0,zip_was_missing,numeric,0.0,2,0.093182,1.787276,0.110822,0.405053,0.002044,33.0,4.0,1.0,1.0,9.75,0.102564,kept,kept,0.956343,True,True,True,True,True,1
1,merchant_state_was_missing,numeric,0.0,2,0.076235,1.555636,0.082347,0.342534,0.000361,36.0,5.0,3.0,2.0,11.50,0.086957,kept,kept,0.149073,True,True,True,True,True,2
2,is_swipe_transaction,numeric,0.0,2,0.074620,1.998686,0.029374,0.192064,0.002331,37.0,2.0,9.0,3.0,12.75,0.078431,kept,kept,0.639524,True,True,True,True,True,3
3,uc_mcc_tx_count_hist,numeric,0.0,8975,0.065017,2.748544,0.036759,0.081699,0.006094,39.0,1.0,8.0,6.0,13.50,0.074074,kept,kept,0.892711,True,True,True,True,True,4
4,new_merchant_for_card_flag,numeric,0.0,2,0.070683,0.552430,0.064075,0.148005,0.003096,38.0,15.0,4.0,4.0,15.25,0.065574,kept,kept,0.237544,True,True,True,True,True,5
5,uc_merchant_tx_count_hist,numeric,0.0,7344,0.099574,1.075460,0.104614,0.019321,0.002411,32.0,7.0,2.0,24.0,16.25,0.061538,kept,kept,0.645554,True,True,True,True,True,6
6,uc_zip_tx_count_hist,numeric,0.0,12624,0.043553,1.262842,0.040973,0.050351,0.004670,45.0,6.0,7.0,11.0,17.25,0.057971,kept,kept,0.931334,True,True,True,True,True,7
7,uc_amount_mean_3m,numeric,0.0,163764,0.168752,0.626893,0.002146,0.094308,0.003483,4.0,13.0,82.0,5.0,26.00,0.038462,kept,kept,0.650977,True,True,True,True,True,8
8,amount_abs,numeric,0.0,25497,0.041908,0.280523,0.007278,0.040962,0.000173,46.0,33.0,21.0,15.0,28.75,0.034783,kept,kept,NaN,True,True,True,True,True,9
9,amount_minus_6m_mean,numeric,0.0,247626,0.036459,0.256098,0.010286,0.058201,0.000288,59.0,39.0,15.0,10.0,30.75,0.032520,kept,kept,0.979360,True,True,True,True,True,10


## Reproducción del split temporal


In [7]:
caps = compute_train_validation_caps(split_summary, max_rows=inputs["train_validation_row_cap"])

effective_fit_summary = pd.DataFrame(
    [
        {
            "split_for_fit": "train_effective",
            "n_rows_full_split": caps["train_rows_full"],
            "rows_used_in_final_fit": caps["train_cap"],
        },
        {
            "split_for_fit": "validation_effective",
            "n_rows_full_split": caps["validation_rows_full"],
            "rows_used_in_final_fit": caps["validation_cap"],
        },
        {
            "split_for_fit": "train_plus_validation_effective",
            "n_rows_full_split": caps["train_validation_rows_full"],
            "rows_used_in_final_fit": inputs["train_validation_row_cap"],
        },
        {
            "split_for_fit": "test_full",
            "n_rows_full_split": int(split_summary.loc[split_summary["split"] == "test", "n_rows"].iloc[0]),
            "rows_used_in_final_fit": int(split_summary.loc[split_summary["split"] == "test", "n_rows"].iloc[0]),
        },
    ]
)

display(split_summary)
display(effective_fit_summary)


,split,n_rows,n_positive,fraud_rate,start_period,end_period
0,train,20604847,25179,0.001222,1991-01,2017-12
1,validation,1721615,2491,0.001447,2018-01,2018-12
2,test,1435413,2087,0.001454,2019-01,2019-10
3,excluded,625025,0,0.000000,2019-11,2020-02


,split_for_fit,n_rows_full_split,rows_used_in_final_fit
0,train_effective,20604847,461445
1,validation_effective,1721615,38555
2,train_plus_validation_effective,22326462,500000
3,test_full,1435413,1435413


## Preparación de X e y


In [8]:
train_frame = load_final_split(
    config,
    "train",
    inputs["selected_features"],
    max_rows=caps["train_cap"],
    sort_by=None,
)
validation_frame = load_final_split(
    config,
    "validation",
    inputs["selected_features"],
    max_rows=caps["validation_cap"],
    sort_by=None,
)
test_frame = load_final_split(
    config,
    "test",
    inputs["selected_features"],
    max_rows=None,
    sort_by=None,
)

X_train = train_frame.loc[:, inputs["selected_features"]].copy()
y_train = train_frame.loc[:, config.target_col].copy()
X_val = validation_frame.loc[:, inputs["selected_features"]].copy()
y_val = validation_frame.loc[:, config.target_col].copy()
X_test = test_frame.loc[:, inputs["selected_features"]].copy()
y_test = test_frame.loc[:, config.target_col].copy()

frame_shapes = pd.DataFrame(
    [
        {"split": "train_effective", "n_rows": len(X_train), "n_features": X_train.shape[1], "n_frauds": int(y_train.sum()), "fraud_rate": float(y_train.mean())},
        {"split": "validation_effective", "n_rows": len(X_val), "n_features": X_val.shape[1], "n_frauds": int(y_val.sum()), "fraud_rate": float(y_val.mean())},
        {"split": "test_full", "n_rows": len(X_test), "n_features": X_test.shape[1], "n_frauds": int(y_test.sum()), "fraud_rate": float(y_test.mean())},
    ]
)

display(frame_shapes)


,split,n_rows,n_features,n_frauds,fraud_rate
0,train_effective,461445,100,25179,0.054566
1,validation_effective,38555,100,2491,0.064609
2,test_full,1435413,100,2087,0.001454


## Construcción del modelo final XGBoost


In [9]:
source_model_artifact = inputs["source_artifact"]
final_model_artifact = joblib.load(BEST_MODEL_FINAL_ARTIFACT_PATH)
final_model = final_model_artifact["model"]
threshold_final = float(final_model_artifact["threshold_final"])

configuration_reference_df = pd.DataFrame(
    [
        {
            "official_final_artifact": BEST_MODEL_FINAL_ARTIFACT_PATH.name,
            "source_experiment_artifact": SOURCE_MODEL_ARTIFACT_PATH.name,
            "model_name": metadata["model_name"],
            "feature_subset": metadata["feature_subset"],
            "balancing_strategy": metadata["balancing_strategy"],
            "random_state": inputs["random_state"],
            "threshold_final": threshold_final,
            "number_of_features_used": len(inputs["selected_features"]),
        }
    ]
)

display(
    Markdown(
        '''
        **Decisión de reconstrucción**

        Notebook 4 ya reentrenó el modelo final en `train + validation` y guardó el artefacto resultante en `best_model_final.joblib`. Para preservar exactamente las métricas finales ya reportadas en la tesis, este notebook usa ese artefacto final como objeto oficial de evaluación.

        Adicionalmente, en la siguiente sección se ejecuta un reentrenamiento de referencia con la misma configuración únicamente para medir tiempo de entrenamiento y documentar el `scale_pos_weight` efectivo aplicado al conjunto final de ajuste. Ese rerun no reemplaza el artefacto final oficial.
        '''
    )
)

display(configuration_reference_df)



        **Decisión de reconstrucción**

        Notebook 4 ya reentrenó el modelo final en `train + validation` y guardó el artefacto resultante en `best_model_final.joblib`. Para preservar exactamente las métricas finales ya reportadas en la tesis, este notebook usa ese artefacto final como objeto oficial de evaluación.

        Adicionalmente, en la siguiente sección se ejecuta un reentrenamiento de referencia con la misma configuración únicamente para medir tiempo de entrenamiento y documentar el `scale_pos_weight` efectivo aplicado al conjunto final de ajuste. Ese rerun no reemplaza el artefacto final oficial.
        

,official_final_artifact,source_experiment_artifact,model_name,feature_subset,balancing_strategy,random_state,threshold_final,number_of_features_used
0,best_model_final.joblib,xgboost_top_100_scale_pos_weight_manual.joblib,xgboost,top_100,scale_pos_weight_manual,42,0.71,100


## Entrenamiento final


In [10]:
train_validation_fit_frame = pd.concat([train_frame, validation_frame], axis=0, ignore_index=True)

timing_rerun = train_final_xgboost_model(
    source_artifact=source_model_artifact,
    train_frame=train_validation_fit_frame,
    feature_columns=inputs["selected_features"],
    target_col=config.target_col,
)

timing_rerun_df = pd.DataFrame(
    [
        {
            "train_rows_used": timing_rerun["train_rows_used"],
            "train_positive_rows": timing_rerun["train_positive_rows"],
            "effective_scale_pos_weight": timing_rerun["effective_scale_pos_weight"],
            "train_time_seconds_reference_rerun": timing_rerun["train_time_seconds"],
        }
    ]
)

display(timing_rerun_df)


,train_rows_used,train_positive_rows,effective_scale_pos_weight,train_time_seconds_reference_rerun
0,500000,27670,17.070112,88.222797


## Predicción sobre test


In [11]:
prediction_bundle = predict_final_model(
    model=final_model,
    frame=test_frame,
    feature_columns=inputs["selected_features"],
)

y_test_scores = prediction_bundle["scores"]
predict_time_seconds = prediction_bundle["predict_time_seconds"]
y_test_pred = get_predicted_labels(y_test_scores, threshold=threshold_final)

prediction_summary = pd.DataFrame(
    [
        {
            "threshold_used": threshold_final,
            "predict_time_seconds": predict_time_seconds,
            "n_test_rows": len(y_test),
            "n_predicted_positive": int(y_test_pred.sum()),
        }
    ]
)

display(prediction_summary)


,threshold_used,predict_time_seconds,n_test_rows,n_predicted_positive
0,0.71,17.463266,1435413,4311


## Evaluación final


In [12]:
final_metrics_df = build_final_metrics_table(
    y_true=y_test,
    scores=y_test_scores,
    threshold=threshold_final,
    model_name=metadata["model_name"],
    feature_subset=metadata["feature_subset"],
    balancing_strategy=metadata["balancing_strategy"],
    number_of_features_used=len(inputs["selected_features"]),
    train_time_seconds=timing_rerun["train_time_seconds"],
    predict_time_seconds=predict_time_seconds,
)

confusion_values = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
confusion_matrix_df = pd.DataFrame(
    [
        {"actual_label": "No Fraud", "pred_no_fraud": int(confusion_values[0, 0]), "pred_fraud": int(confusion_values[0, 1])},
        {"actual_label": "Fraud", "pred_no_fraud": int(confusion_values[1, 0]), "pred_fraud": int(confusion_values[1, 1])},
    ]
)

classification_report_df = (
    pd.DataFrame(classification_report(y_test, y_test_pred, output_dict=True, zero_division=0))
    .T.reset_index().rename(columns={"index": "class_or_average"})
)

final_metrics_row = final_metrics_df.iloc[0].to_dict()

summary_reference = inputs["final_metrics_summary"].copy()
metric_col = summary_reference.columns[0]
value_col = summary_reference.columns[1]
summary_reference[value_col] = summary_reference[value_col].astype(float)
metric_mapping = {
    "PR-AUC": final_metrics_row["pr_auc"],
    "ROC-AUC": final_metrics_row["roc_auc"],
    "Precision": final_metrics_row["precision"],
    "Recall": final_metrics_row["recall"],
    "F1-Score": final_metrics_row["f1_score"],
    "Accuracy": final_metrics_row["accuracy"],
}
consistency_check = summary_reference[[metric_col, value_col]].rename(columns={metric_col: "metric_name", value_col: "notebook4_value"})
consistency_check["notebook5_value"] = consistency_check["metric_name"].map(metric_mapping)
consistency_check["absolute_difference"] = (consistency_check["notebook4_value"] - consistency_check["notebook5_value"]).abs()

if not (consistency_check["absolute_difference"] < 1e-6).all():
    raise AssertionError("Las métricas de notebook 5 no reproducen exactamente el artefacto final de notebook 4.")

display(final_metrics_df)
display(confusion_matrix_df)
display(classification_report_df)
display(consistency_check)


,model_name,feature_subset,balancing_strategy,number_of_features_used,threshold_used,accuracy,precision,recall,f1_score,roc_auc,pr_auc,tp,fp,tn,fn,n_obs,n_positive,positive_rate,train_time_seconds,predict_time_seconds
0,xgboost,top_100,scale_pos_weight_manual,100,0.71,0.997895,0.391556,0.808816,0.527665,0.998689,0.660748,1688,2623,1430703,399,1435413,2087,0.001454,88.222797,17.463266


,actual_label,pred_no_fraud,pred_fraud
0,No Fraud,1430703,2623
1,Fraud,399,1688


,class_or_average,precision,recall,f1-score,support
0,0,0.999721,0.998170,0.998945,1.433326e+06
1,1,0.391556,0.808816,0.527665,2.087000e+03
2,accuracy,0.997895,0.997895,0.997895,9.978947e-01
3,macro avg,0.695639,0.903493,0.763305,1.435413e+06
4,weighted avg,0.998837,0.997895,0.998260,1.435413e+06


,metric_name,notebook4_value,notebook5_value,absolute_difference
0,PR-AUC,0.660748,0.660748,1.871567e-07
1,ROC-AUC,0.998689,0.998689,1.449660e-08
2,Precision,0.391556,0.391556,4.834145e-07
3,Recall,0.808816,0.808816,4.829899e-07
4,F1-Score,0.527665,0.527665,1.047202e-07
5,Accuracy,0.997895,0.997895,3.174243e-07


## Figuras finales


In [13]:
plot_confusion_matrix(
    confusion_values=confusion_values,
    labels=("No Fraud", "Fraud"),
    title=f"Matriz de Confusión - {metadata['model_name']}_{metadata['feature_subset']}\n(Threshold={threshold_final:.2f})",
    output_path=FIGURES_DIR / "confusion_matrix_best_model.png",
)
plot_roc_curve(
    y_true=y_test,
    scores=y_test_scores,
    model_name=f"{metadata['model_name']}_{metadata['feature_subset']}",
    output_path=FIGURES_DIR / "roc_curve_best_model.png",
)
plot_precision_recall_curve(
    y_true=y_test,
    scores=y_test_scores,
    model_name=f"{metadata['model_name']}_{metadata['feature_subset']}",
    output_path=FIGURES_DIR / "precision_recall_curve_best_model.png",
)
plot_probability_distribution(
    scores=y_test_scores,
    y_true=y_test,
    threshold=threshold_final,
    title="Distribución de probabilidades - Modelo final XGBoost",
    output_path=FIGURES_DIR / "final_model_probability_distribution.png",
)

feature_importance_df = build_feature_importance_frame(final_model, inputs["selected_features"])
plot_feature_importance(
    importance_frame=feature_importance_df,
    top_n=30,
    title="Importancia de variables - Top 30 del modelo final XGBoost",
    output_path=FIGURES_DIR / "final_model_feature_importance_top_30.png",
)

display(feature_importance_df.head(30))


,feature,importance
0,zip_was_missing,0.354791
1,uc_merchant_tx_count_hist,0.090254
2,new_merchant_for_card_flag,0.075519
3,merchant_state_was_missing,0.028398
4,new_zip_for_card_flag,0.023968
5,uc_zip_tx_count_hist,0.021500
6,uc_online_tx_count_3m,0.021489
7,uc_days_since_prev_tx,0.020322
8,is_chip_transaction,0.018913
9,uc_mcc_tx_count_hist,0.016950


## Tablas finales


In [14]:
configuration_df = build_configuration_table(
    config=config,
    inputs=inputs,
    effective_scale_pos_weight=timing_rerun["effective_scale_pos_weight"],
)

save_table_with_optional_excel(final_metrics_df, TABLES_DIR / "final_model_test_metrics.csv", TABLES_DIR / "final_model_test_metrics.xlsx")
save_table_with_optional_excel(confusion_matrix_df, TABLES_DIR / "final_model_confusion_matrix.csv", TABLES_DIR / "final_model_confusion_matrix.xlsx")
save_table_with_optional_excel(classification_report_df, TABLES_DIR / "final_model_classification_report.csv", TABLES_DIR / "final_model_classification_report.xlsx")
save_table_with_optional_excel(selected_features_final, TABLES_DIR / "final_model_selected_features.csv", TABLES_DIR / "final_model_selected_features.xlsx")
save_table_with_optional_excel(configuration_df, TABLES_DIR / "final_model_configuration.csv", TABLES_DIR / "final_model_configuration.xlsx")

generated_tables = pd.DataFrame(
    {
        "table_path": [
            "outputs/tables/final_model_test_metrics.csv",
            "outputs/tables/final_model_confusion_matrix.csv",
            "outputs/tables/final_model_classification_report.csv",
            "outputs/tables/final_model_selected_features.csv",
            "outputs/tables/final_model_configuration.csv",
        ]
    }
)
display(generated_tables)


,table_path
0,outputs/tables/final_model_test_metrics.csv
1,outputs/tables/final_model_confusion_matrix.csv
2,outputs/tables/final_model_classification_repo...
3,outputs/tables/final_model_selected_features.csv
4,outputs/tables/final_model_configuration.csv


## Guardado del modelo final


In [15]:
final_model_output_path = MODELS_DIR / "xgboost_top_100_scale_pos_weight_manual_final.joblib"

final_artifact_payload = build_final_model_artifact(
    model=final_model,
    feature_columns=inputs["selected_features"],
    threshold_final=threshold_final,
    hyperparameters=hyperparameters,
    random_state=inputs["random_state"],
    final_metrics_row=final_metrics_row,
    configuration_row=configuration_df.iloc[0].to_dict(),
)
final_artifact_payload["data_paths_used"] = {
    "modeling_dataset_path": str(MODELING_DATASET_PATH),
    "selected_features_path": str(SELECTED_FEATURES_PATH),
    "source_model_artifact_path": str(SOURCE_MODEL_ARTIFACT_PATH),
    "best_model_final_artifact_path": str(BEST_MODEL_FINAL_ARTIFACT_PATH),
}
final_artifact_payload["timing_reference_rerun"] = timing_rerun_df.iloc[0].to_dict()

joblib.dump(final_artifact_payload, final_model_output_path)

display(pd.DataFrame([{"saved_model_path": str(final_model_output_path), "exists": final_model_output_path.exists()}]))


,saved_model_path,exists
0,C:\Users\cecor\OneDrive - Universidad Nacional...,True


## Resumen para tesis


In [16]:
thesis_summary_markdown = f'''
**Resumen académico del modelo final**

El modelo final seleccionado para la detección de fraude transaccional con tarjetas corresponde a un **XGBoost** entrenado sobre el subconjunto de variables **top_100**, con estrategia de balanceo **scale_pos_weight_manual** y umbral final de decisión **{threshold_final:.2f}**, definido exclusivamente con el conjunto de **validation** en notebook 4.

En la evaluación final sobre **test**, el modelo obtuvo las siguientes métricas: **accuracy = {final_metrics_row["accuracy"]:.4f}**, **precision = {final_metrics_row["precision"]:.4f}**, **recall = {final_metrics_row["recall"]:.4f}**, **F1-score = {final_metrics_row["f1_score"]:.4f}**, **PR-AUC = {final_metrics_row["pr_auc"]:.4f}** y **ROC-AUC = {final_metrics_row["roc_auc"]:.4f}**.

Desde una perspectiva interpretativa, la **precision** indica la proporción de alertas de fraude que efectivamente corresponden a fraude real, mientras que el **recall** muestra la capacidad del modelo para recuperar fraudes verdaderos. El **F1-score** resume el equilibrio entre precision y recall, la **PR-AUC** es la métrica principal por tratarse de un problema altamente desbalanceado, y la **ROC-AUC** describe la capacidad global de discriminación del modelo.

Metodológicamente, es fundamental subrayar que el conjunto **test** se utilizó exclusivamente para la **evaluación final** del modelo ya seleccionado; no participó en selección de variables, ajuste de hiperparámetros ni optimización del threshold.
'''

display(Markdown(thesis_summary_markdown))



**Resumen académico del modelo final**

El modelo final seleccionado para la detección de fraude transaccional con tarjetas corresponde a un **XGBoost** entrenado sobre el subconjunto de variables **top_100**, con estrategia de balanceo **scale_pos_weight_manual** y umbral final de decisión **0.71**, definido exclusivamente con el conjunto de **validation** en notebook 4.

En la evaluación final sobre **test**, el modelo obtuvo las siguientes métricas: **accuracy = 0.9979**, **precision = 0.3916**, **recall = 0.8088**, **F1-score = 0.5277**, **PR-AUC = 0.6607** y **ROC-AUC = 0.9987**.

Desde una perspectiva interpretativa, la **precision** indica la proporción de alertas de fraude que efectivamente corresponden a fraude real, mientras que el **recall** muestra la capacidad del modelo para recuperar fraudes verdaderos. El **F1-score** resume el equilibrio entre precision y recall, la **PR-AUC** es la métrica principal por tratarse de un problema altamente desbalanceado, y la **ROC-AUC** describe la capacidad global de discriminación del modelo.

Metodológicamente, es fundamental subrayar que el conjunto **test** se utilizó exclusivamente para la **evaluación final** del modelo ya seleccionado; no participó en selección de variables, ajuste de hiperparámetros ni optimización del threshold.


## Conclusiones del notebook


In [17]:
conclusions_markdown = '''
**Conclusiones**

Este notebook cierra la evaluación empírica del modelo final de la tesis. Las decisiones de modelo, subconjunto de variables, estrategia de balanceo, reentrenamiento metodológico y threshold final provienen del notebook 4 y se respetan aquí sin introducir una nueva etapa de selección.

El objetivo principal de este notebook es dejar una reconstrucción clara del modelo final XGBoost, junto con sus métricas, tablas, figuras y artefactos exportables para el capítulo 5 de la tesis. Los archivos generados en `outputs/tables`, `outputs/figures` y `outputs/models` deben utilizarse como insumo directo para la redacción de resultados y conclusiones.
'''

generated_outputs = pd.DataFrame(
    {
        "artifact": [
            "notebooks/5_model_final_cards.ipynb",
            "outputs/tables/final_model_test_metrics.csv",
            "outputs/tables/final_model_confusion_matrix.csv",
            "outputs/tables/final_model_classification_report.csv",
            "outputs/tables/final_model_selected_features.csv",
            "outputs/tables/final_model_configuration.csv",
            "outputs/figures/confusion_matrix_best_model.png",
            "outputs/figures/roc_curve_best_model.png",
            "outputs/figures/precision_recall_curve_best_model.png",
            "outputs/figures/final_model_probability_distribution.png",
            "outputs/figures/final_model_feature_importance_top_30.png",
            "outputs/models/xgboost_top_100_scale_pos_weight_manual_final.joblib",
        ]
    }
)

display(Markdown(conclusions_markdown))
display(generated_outputs)



**Conclusiones**

Este notebook cierra la evaluación empírica del modelo final de la tesis. Las decisiones de modelo, subconjunto de variables, estrategia de balanceo, reentrenamiento metodológico y threshold final provienen del notebook 4 y se respetan aquí sin introducir una nueva etapa de selección.

El objetivo principal de este notebook es dejar una reconstrucción clara del modelo final XGBoost, junto con sus métricas, tablas, figuras y artefactos exportables para el capítulo 5 de la tesis. Los archivos generados en `outputs/tables`, `outputs/figures` y `outputs/models` deben utilizarse como insumo directo para la redacción de resultados y conclusiones.


,artifact
0,notebooks/5_model_final_cards.ipynb
1,outputs/tables/final_model_test_metrics.csv
2,outputs/tables/final_model_confusion_matrix.csv
3,outputs/tables/final_model_classification_repo...
4,outputs/tables/final_model_selected_features.csv
5,outputs/tables/final_model_configuration.csv
6,outputs/figures/confusion_matrix_best_model.png
7,outputs/figures/roc_curve_best_model.png
8,outputs/figures/precision_recall_curve_best_mo...
9,outputs/figures/final_model_probability_distri...
